# Parte 0.b — El fragmento que se corta a 128

Este notebook genera la evidencia que el proyecto no imprime por sí solo: fuerza una ingesta con `chunk_tokens=900` y compara el embedding de un texto largo contra el embedding de su versión recortada.

Ejecuta el notebook con el kernel de `.venv`. La Parte 0 usa MiniLM en modo local, sin VPN, API ni Docker.

## Evidencia 0.b.1 — Aviso de ingesta

Esta celda ingiere `ejemplos/` pidiendo fragmentos de 900 tokens. MiniLM tiene un máximo de 128 tokens, por lo que debe aparecer el aviso de truncamiento. Copia esa salida al informe.

In [ ]:
import os

# Debe ejecutarse antes de importar rag_pipeline.
os.environ['EMBEDDING_BACKEND'] = 'local'
os.environ['QDRANT_URL'] = ':memory:'

from rag_pipeline import RagPipeline

pipeline = RagPipeline()
chunks = pipeline.ingest('ejemplos', chunk_tokens=900, overlap_tokens=180)
print(f'Fragmentos producidos: {len(chunks)}')

: 

## Evidencia 0.b.2 — Embedding completo vs. recortado

Antes de ejecutar esta celda, crea `texto_900.txt` en la raíz del proyecto con aproximadamente 900 palabras de texto real y con sentido (no el PDF escaneado). La celda cuenta sus tokens y calcula el coseno entre el vector del texto completo y el de sus primeros 126 tokens.

Un coseno cercano a 1 demuestra que el codificador truncó el texto completo de forma silenciosa: para MiniLM, el contenido posterior al límite no influye en el vector.

In [ ]:
from pathlib import Path
import numpy as np

texto_path = Path('texto_900.txt')
if not texto_path.exists():
    raise FileNotFoundError(
        'Crea texto_900.txt en la raíz del proyecto con unas 900 palabras y vuelve a ejecutar.'
    )

texto = texto_path.read_text(encoding='utf-8')
ids = pipeline.encoder.tokenizer(
    texto, add_special_tokens=False, truncation=False
)["input_ids"]

tope_modelo = pipeline.encoder.max_seq_tokens
tope_efectivo = tope_modelo - 2  # dos posiciones se reservan para tokens especiales
texto_recortado = pipeline.encoder.tokenizer.decode(ids[:tope_efectivo])

vector_completo = pipeline.encoder.encode([texto])[0]
vector_recortado = pipeline.encoder.encode([texto_recortado])[0]
coseno = float(np.dot(vector_completo, vector_recortado))

print(f'Palabras del texto: {len(texto.split())}')
print(f'Tokens del texto: {len(ids)}')
print(f'Tope del modelo: {tope_modelo}')
print(f'Tokens efectivos antes de especiales: {tope_efectivo}')
print(f'Coseno texto completo vs. recortado: {coseno:.6f}')

## Qué reportar

Para el entregable 0.b pega el aviso de la primera celda y los números de la segunda: tokens del texto, tope de 128 y coseno. Después declara el valor de fragmentación de la Parte 1 y justifícalo. Una decisión inicial razonable con `bge-m3` es `chunk_tokens=512` y `overlap_tokens=102`: cabe ampliamente bajo su máximo de 8192 tokens, sin convertir cada fragmento en un documento completo.